In [1]:
import os
import re
import json
import time
import requests

In [2]:
os.environ["SCRAPERAPI_KEY"] = "fe3f5ba5f4aa0db8f2175ab3afd80341"  # paste your real key here for this session only

In [3]:
with open("carrefour_FKEN1700000.html", "r", encoding="utf-8") as f:
    html = f.read()

print("Total HTML length:", len(html))

# Check for other common embedded-data patterns
for marker in ["__NUXT__", "window.__INITIAL_STATE__", "application/ld+json", 
               "\"products\"", "\"price\"", "self.__next_f", "window.__APOLLO_STATE__"]:
    print(f"{marker!r} found: {marker in html}")

Total HTML length: 1220076
'__NUXT__' found: False
'window.__INITIAL_STATE__' found: False
'application/ld+json' found: False
'"products"' found: False
'"price"' found: False
'self.__next_f' found: True
'window.__APOLLO_STATE__' found: False


In [4]:
# Find all self.__next_f.push([...]) calls
pattern = re.compile(r'self\.__next_f\.push\(\[(\d+),(".*?")\]\)', re.DOTALL)
matches = pattern.findall(html)

print(f"Found {len(matches)} chunks")

# Search chunks for anything that looks like product data
for idx, (chunk_id, chunk_str) in enumerate(matches):
    if '"price"' in chunk_str or "'price'" in chunk_str or "price" in chunk_str.lower():
        print(f"--- Chunk {idx} (id={chunk_id}) contains 'price' ---")
        print(chunk_str[:500])
        print()

Found 34 chunks
--- Chunk 7 (id=1) contains 'price' ---
"15:[\"$\",\"$L16\",null,{\"locale\":\"en\",\"now\":\"$D2026-09-21T06:31:27.396Z\",\"timeZone\":\"UTC\",\"messages\":{\"address\":{\"add\":\"Add\",\"deliveryAddress\":\"Delivery Address\",\"change\":\"Change\",\"addAnAddress\":\"Add an address to deliver to!\",\"deliverTo\":\"Deliver to:\",\"addNewAddress\":\"Add New Address\",\"changeAddress\":\"Change Address\",\"noAddressMessage\":\"You haven't added any delivery addresses yet. Add an address to enjoy seamless shopping and quick deliveries!\"

--- Chunk 18 (id=1) contains 'price' ---
"d:[\"$\",\"$L1c\",null,{\"availableCountries\":[{\"code\":\"AE\",\"image\":\"https://mafretailblobstorage.blob.core.windows.net/countryimages/AE.png\",\"nameAr\":\"الإمارات العربية المتحدة\",\"nameEn\":\"United Arab Emirates\",\"name\":\"United Arab Emirates\",\"phoneCountryCode\":\"+971\",\"storeId\":\"mafuae\",\"languages\":[{\"code\":\"en\",\"name\":\"English\",\"isDefault\":true},{\"code\":\"a

In [5]:
# Alternative: just find raw text chunks between self.__next_f.push([ and the matching ])
chunks = re.findall(r'self\.__next_f\.push\(\[\d+,(.*?)\]\)\s*</script>', html, re.DOTALL)
print(f"Found {len(chunks)} chunks (alt method)")

for i, c in enumerate(chunks):
    if "price" in c.lower():
        print(f"--- Chunk {i} ---")
        print(c[:800])
        print()

Found 35 chunks (alt method)
--- Chunk 8 ---
"15:[\"$\",\"$L16\",null,{\"locale\":\"en\",\"now\":\"$D2026-09-21T06:31:27.396Z\",\"timeZone\":\"UTC\",\"messages\":{\"address\":{\"add\":\"Add\",\"deliveryAddress\":\"Delivery Address\",\"change\":\"Change\",\"addAnAddress\":\"Add an address to deliver to!\",\"deliverTo\":\"Deliver to:\",\"addNewAddress\":\"Add New Address\",\"changeAddress\":\"Change Address\",\"noAddressMessage\":\"You haven't added any delivery addresses yet. Add an address to enjoy seamless shopping and quick deliveries!\"},\"addressJourney\":{\"addAddressDetails\":\"Add address details\",\"addDeliveryLocation\":\"Add delivery location\",\"addressForm\":{\"addressType\":\"Address type*\",\"appartment\":\"Apartment/villa number*\",\"area\":\"Area\",\"block\":\"Block*\",\"building\":\"Building*\",\"city\":\"City*\",\"c

--- Chunk 19 ---
"d:[\"$\",\"$L1c\",null,{\"availableCountries\":[{\"code\":\"AE\",\"image\":\"https://mafretailblobstorage.blob.core.windows.net/country

In [6]:
# Print much more of chunk 32 to see the actual product listing structure
print(len(chunks[32]))
print(chunks[32][:5000])

1695
"62:I[52563,[\"2044\",\"static/chunks/891cff7f-0a93e1bcdb8ca72d.js\",\"41\",\"static/chunks/41-bc835a536ba83ba0.js\",\"1661\",\"static/chunks/1661-daa67cd50a8a34ab.js\",\"294\",\"static/chunks/294-5fc80f3bf5494a6f.js\",\"8295\",\"static/chunks/8295-26c391e78d22462a.js\",\"6340\",\"static/chunks/6340-bcfb80012d8717c2.js\",\"7787\",\"static/chunks/7787-a1d7fc36bb99ac20.js\",\"7905\",\"static/chunks/7905-591b97064ae470bd.js\",\"5648\",\"static/chunks/5648-f4d4c1b602715cbc.js\",\"7987\",\"static/chunks/7987-23fc1ab1b716e50c.js\",\"803\",\"static/chunks/803-5f8d4c70b94fe497.js\",\"3825\",\"static/chunks/3825-1756214479722ce2.js\",\"833\",\"static/chunks/833-26aa4ec3be54f114.js\",\"6276\",\"static/chunks/6276-dcfcbd66a1cf7f6c.js\",\"3997\",\"static/chunks/3997-ef0233e911a8bc94.js\",\"324\",\"static/chunks/324-0cc7ee718c61c441.js\",\"7378\",\"static/chunks/7378-139249c86315fa1b.js\",\"5816\",\"static/chunks/5816-41b0e374bb548e1c.js\",\"3864\",\"static/chunks/3864-4815e2ecad751979.js\",\"

In [7]:
# Search all chunks for more specific product-data indicators
for i, c in enumerate(chunks):
    if any(key in c for key in ['"sku"', '"productName"', '"title"', '"sellingPrice"', 
                                   '"listPrice"', '"offers"', '"itemName"']):
        print(f"--- Chunk {i} matches a stronger product indicator ---")
        print(c[:300])
        print()

In [8]:
with open("chunk_32_full.txt", "w", encoding="utf-8") as f:
    f.write(chunks[32])
print("Saved chunk 32, length:", len(chunks[32]))

Saved chunk 32, length: 1695


In [9]:
# Show size of every chunk, sorted largest first
sizes = [(i, len(c)) for i, c in enumerate(chunks)]
sizes.sort(key=lambda x: -x[1])
for i, size in sizes[:10]:
    print(f"Chunk {i}: {size} chars")

Chunk 33: 390470 chars
Chunk 19: 146990 chars
Chunk 7: 87993 chars
Chunk 8: 26882 chars
Chunk 30: 10360 chars
Chunk 28: 8510 chars
Chunk 3: 2492 chars
Chunk 29: 2354 chars
Chunk 17: 2273 chars
Chunk 15: 2266 chars


In [10]:
biggest_idx = sizes[0][0]
with open(f"chunk_{biggest_idx}_full.txt", "w", encoding="utf-8") as f:
    f.write(chunks[biggest_idx])
print(f"Saved chunk {biggest_idx}, length: {sizes[0][1]}")

Saved chunk 33, length: 390470


In [11]:
# Load the saved chunk and look at the beginning to understand its structure
with open("chunk_33_full.txt", "r", encoding="utf-8") as f:
    chunk33 = f.read()

print(chunk33[:2000])

"54:[[\"$\",\"div\",null,{\"aria-hidden\":true,\"className\":\"hidden\",\"children\":[[\"$\",\"a\",\"Next\",{\"href\":\"/mafken/en/c/FKEN1700000?currentPage=1\",\"rel\":\"next\",\"children\":\"Next\"}]]}],[\"$\",\"$L62\",null,{\"adTechData\":{\"banners\":{},\"storyAds\":{},\"carousel\":{},\"flagship\":{},\"sponsoredProduct\":{},\"video\":{},\"brandShelves\":{},\"brandShelvesLiteData\":{}},\"categoryId\":\"FKEN1700000\",\"data\":{\"topStickySections\":[{\"uid\":\"refinement-bar\",\"rowId\":\"b339e5f5-bd77-45eb-a3be-1eda7c9d6254\",\"componentDTO\":{\"refinementComponents\":[{\"uid\":\"filters\",\"rowId\":\"22c5b1e5-60c5-4791-a9b6-cfcf9e7fc130\",\"componentDTO\":{\"title\":\"Filters\",\"filterAppliedCount\":0,\"filters\":[{\"code\":\"deal\",\"name\":\"Deals\",\"multiSelect\":false,\"values\":[{\"code\":true,\"id\":true,\"name\":\"Today's Deals\",\"selected\":false}]},{\"code\":\"brand_name\",\"name\":\"Brand\",\"multiSelect\":true,\"values\":[{\"code\":\"'TROPICAL HEAT'\",\"id\":\"'TROPIC

In [12]:
# Find all occurrences of "price" and show surrounding context
for m in re.finditer(r'"price"', chunk33):
    start = max(0, m.start() - 100)
    end = min(len(chunk33), m.end() + 200)
    print(chunk33[start:end])
    print("---")

In [13]:
# Case-insensitive, no quote requirement
import re

for keyword in ["price", "Price", "amount", "cost", "brand_name", "productName", 
                 "sku", "title", "name\\\":\\\"", "sellingPrice"]:
    count = chunk33.lower().count(keyword.lower())
    print(f"{keyword!r}: {count} occurrences")

'price': 384 occurrences
'Price': 384 occurrences
'amount': 0 occurrences
'cost': 0 occurrences
'brand_name': 1 occurrences
'productName': 62 occurrences
'sku': 0 occurrences
'title': 445 occurrences
'name\\":\\"': 472 occurrences
'sellingPrice': 62 occurrences


In [14]:
# Print further into the chunk, past the filters section we already saw
print(chunk33[2000:5000])

false},{\"code\":\"'PEPTANG'\",\"id\":\"'PEPTANG'\",\"name\":\"PEPTANG\",\"selected\":false},{\"code\":\"'CLOVERS'\",\"id\":\"'CLOVERS'\",\"name\":\"CLOVERS\",\"selected\":false},{\"code\":\"'AMIGOS'\",\"id\":\"'AMIGOS'\",\"name\":\"AMIGOS\",\"selected\":false},{\"code\":\"'NUTRAMEAL'\",\"id\":\"'NUTRAMEAL'\",\"name\":\"NUTRAMEAL\",\"selected\":false},{\"code\":\"'SUNVEAT'\",\"id\":\"'SUNVEAT'\",\"name\":\"SUNVEAT\",\"selected\":false},{\"code\":\"'CRF'\",\"id\":\"'CRF'\",\"name\":\"CRF\",\"selected\":false},{\"code\":\"'SHAN'\",\"id\":\"'SHAN'\",\"name\":\"SHAN\",\"selected\":false},{\"code\":\"'SIMBA MBILI'\",\"id\":\"'SIMBA MBILI'\",\"name\":\"SIMBA MBILI\",\"selected\":false},{\"code\":\"'SANTA MARIA'\",\"id\":\"'SANTA MARIA'\",\"name\":\"SANTA MARIA\",\"selected\":false},{\"code\":\"'BAKERS'\",\"id\":\"'BAKERS'\",\"name\":\"BAKERS\",\"selected\":false},{\"code\":\"'LP - NONAME'\",\"id\":\"'LP - NONAME'\",\"name\":\"LP - NONAME\",\"selected\":false},{\"code\":\"'WOTS'\",\"id\":\"'W

In [15]:
# "TROPICAL HEAT" was one of the brand filter values -- search where else it appears,
# since an actual product entry would also reference its brand
idx = chunk33.find("TROPICAL HEAT", 200)  # skip the first occurrence in the filters block
print(idx)
if idx != -1:
    print(chunk33[idx-200:idx+500])

967
\"multiSelect\":false,\"values\":[{\"code\":true,\"id\":true,\"name\":\"Today's Deals\",\"selected\":false}]},{\"code\":\"brand_name\",\"name\":\"Brand\",\"multiSelect\":true,\"values\":[{\"code\":\"'TROPICAL HEAT'\",\"id\":\"'TROPICAL HEAT'\",\"name\":\"TROPICAL HEAT\",\"selected\":false},{\"code\":\"'ZESTA'\",\"id\":\"'ZESTA'\",\"name\":\"ZESTA\",\"selected\":false},{\"code\":\"'NUVITA'\",\"id\":\"'NUVITA'\",\"name\":\"NUVITA\",\"selected\":false},{\"code\":\"'KAPUTEI'\",\"id\":\"'KAPUTEI'\",\"name\":\"KAPUTEI\",\"selected\":false},{\"code\":\"'TOP FOOD'\",\"id\":\"'TOP FOOD'\",\"name\":\"TOP FOOD\",\"selected\":false},{\"code\":\"'Mulsons'\",\"id\":\"'Mulsons'\",\"name\":\"Mulsons\",\"sel


In [16]:
idx = chunk33.find('"productName"')
print(idx)
print(chunk33[max(0, idx-200):idx+800])

-1
"54:[[\"$\",\"div\",null,{\"aria-hidden\":true,\"className\":\"hidden\",\"children\":[[\"$\",\"a\",\"Next\",{\"href\":\"/mafken/en/c/FKEN1700000?currentPage=1\",\"rel\":\"next\",\"children\":\"Next\"}]]}],[\"$\",\"$L62\",null,{\"adTechData\":{\"banners\":{},\"storyAds\":{},\"carousel\":{},\"flagship\":{},\"sponsoredProduct\":{},\"video\":{},\"brandShelves\":{},\"brandShelvesLiteData\":{}},\"categoryId\":\"FKEN1700000\",\"data\":{\"topStickySections\":[{\"uid\":\"refinement-bar\",\"rowId\":\"b339e5f5-bd77-45eb-a3be-1eda7c9d6254\",\"componentDTO\":{\"refinementComponents\":[{\"uid\":\"filters\",\"rowId\":\"22c5b1e5-60c5-4791-a9b6-cfcf9e7fc130\",\"componentDTO\":{\"title\":\"Filters\",\"filterAppliedCount\":0,\"filters\":[{\"code\":\"deal\",\"name\":\"Deals\",\"multiSelect\":false,\"values\"


In [17]:
idx = chunk33.find('productName')
print(idx)
print(chunk33[max(0, idx-200):idx+800])

36532
ageUrl\":\"https://cdn.mafrservices.com/sys-master-root/hd8/hef/17328251371550/136147_main.jpg?im=Resize=(300,300)\",\"intent\":\"SLOTTED\",\"offerId\":\"offer_carrefour_\",\"productId\":\"136147\",\"productName\":\"Wrigley's Juicy Fruit Strawberry Chewing Gum 14g x 10 Pieces\",\"pis\":[\"SLOTTED\"],\"serviceProductCodes\":[\"\"],\"isSoldByWeight\":false,\"shopId\":\"0000\",\"shopName\":\"Carrefour\",\"shippingIndicator\":\"SLOTTED\",\"quantityTypeKey\":\"pieces\",\"sellingPrice\":27,\"markedPrice\":27,\"currency\":\"KES\",\"productType\":\"FOOD\",\"isExpress\":false,\"isSponsored\":true,\"productCategory\":[\"fken1700000\"],\"currencySymbol\":\"\",\"isDirectMatch\":\"false\",\"warranty\":false,\"isQuantityPickerEnabled\":true,\"isAddOnEnabled\":false,\"shouldReplaceQtyEntryInLocal\":false,\"productUrl\":\"/mafken/en/chewing-gum/juicy-fruit-strawberry-gum-10p-14g/p/136147\",\"internationalShipping\":false,\"productCardVersion\":2},\"analytics\":{\"brandName\":\"food_juicy\",\"cam

In [18]:
idx = chunk33.find('quantityTypeKey')
print(chunk33[max(0, idx-300):idx+300])

\"offerId\":\"offer_carrefour_\",\"productId\":\"136147\",\"productName\":\"Wrigley's Juicy Fruit Strawberry Chewing Gum 14g x 10 Pieces\",\"pis\":[\"SLOTTED\"],\"serviceProductCodes\":[\"\"],\"isSoldByWeight\":false,\"shopId\":\"0000\",\"shopName\":\"Carrefour\",\"shippingIndicator\":\"SLOTTED\",\"quantityTypeKey\":\"pieces\",\"sellingPrice\":27,\"markedPrice\":27,\"currency\":\"KES\",\"productType\":\"FOOD\",\"isExpress\":false,\"isSponsored\":true,\"productCategory\":[\"fken1700000\"],\"currencySymbol\":\"\",\"isDirectMatch\":\"false\",\"warranty\":false,\"isQuantityPickerEnabled\":true,\"i


In [19]:
idx = chunk33.find('productId')
print(chunk33[idx:idx+2500])

productId\":\"136147\",\"productName\":\"Wrigley's Juicy Fruit Strawberry Chewing Gum 14g x 10 Pieces\",\"pis\":[\"SLOTTED\"],\"serviceProductCodes\":[\"\"],\"isSoldByWeight\":false,\"shopId\":\"0000\",\"shopName\":\"Carrefour\",\"shippingIndicator\":\"SLOTTED\",\"quantityTypeKey\":\"pieces\",\"sellingPrice\":27,\"markedPrice\":27,\"currency\":\"KES\",\"productType\":\"FOOD\",\"isExpress\":false,\"isSponsored\":true,\"productCategory\":[\"fken1700000\"],\"currencySymbol\":\"\",\"isDirectMatch\":\"false\",\"warranty\":false,\"isQuantityPickerEnabled\":true,\"isAddOnEnabled\":false,\"shouldReplaceQtyEntryInLocal\":false,\"productUrl\":\"/mafken/en/chewing-gum/juicy-fruit-strawberry-gum-10p-14g/p/136147\",\"internationalShipping\":false,\"productCardVersion\":2},\"analytics\":{\"brandName\":\"food_juicy\",\"campaignId\":\"887955401994592256\",\"isOffer\":false,\"links\":{\"images\":[{\"rel\":\"assets\",\"href\":\"https://cdn.mafrservices.com/sys-master-root/hd8/hef/17328251371550/136147_m

In [20]:
# Extract full product objects using productId as anchor
product_pattern = re.compile(
    r'\\"productId\\":\\"(?P<productId>[^"\\]+)\\",\\"productName\\":\\"(?P<productName>[^"\\]+)\\".*?'
    r'\\"sellingPrice\\":(?P<sellingPrice>[\d.]+).*?'
    r'\\"markedPrice\\":(?P<markedPrice>[\d.]+).*?'
    r'\\"currency\\":\\"(?P<currency>[^"\\]+)\\".*?'
    r'\\"productType\\":\\"(?P<productType>[^"\\]+)\\".*?'
    r'\\"productUrl\\":\\"(?P<productUrl>[^"\\]+)\\"',
    re.DOTALL
)

# Regex to pull pack size/quantity out of the product name itself
unit_pattern = re.compile(
    r'(\d+(?:\.\d+)?\s?(?:g|kg|ml|l|Ltrs?|litres?)\s?x?\s?\d*\s?(?:pieces?|pcs?|pack)?)',
    re.IGNORECASE
)

matches = list(product_pattern.finditer(chunk33))
print(f"Matched {len(matches)} products")

products = []
for m in matches:
    d = m.groupdict()
    unit_match = unit_pattern.search(d["productName"])
    d["unit"] = unit_match.group(1).strip() if unit_match else None
    products.append(d)

for p in products[:5]:
    print(p)

Matched 60 products
{'productId': '136147', 'productName': "Wrigley's Juicy Fruit Strawberry Chewing Gum 14g x 10 Pieces", 'sellingPrice': '27', 'markedPrice': '27', 'currency': 'KES', 'productType': 'FOOD', 'productUrl': '/mafken/en/chewing-gum/juicy-fruit-strawberry-gum-10p-14g/p/136147', 'unit': '14g x 10 Pieces'}
{'productId': '37226', 'productName': 'Juicy Fruit Chewing Gum 10pcs 14g', 'sellingPrice': '27', 'markedPrice': '27', 'currency': 'KES', 'productType': 'FOOD', 'productUrl': '/mafken/en/chewing-gum/juicy-fruit-chewing-gum-10p-14g/p/37226', 'unit': '14g'}
{'productId': '85033', 'productName': 'Royco Beef Cube Fortified  4g x 40', 'sellingPrice': '199', 'markedPrice': '199', 'currency': 'KES', 'productType': 'FOOD', 'productUrl': '/mafken/en/seasoning/royco-cube-fortified-4gx40/p/85033', 'unit': '4g x 40'}
{'productId': '185401', 'productName': 'Royco Beef Cube 4g x 28', 'sellingPrice': '140', 'markedPrice': '140', 'currency': 'KES', 'productType': 'FOOD', 'productUrl': '/ma

In [44]:
"""
Carrefour full category scraper using ScraperAPI + regex extraction.

Setup:
    pip install requests beautifulsoup4

Usage (Jupyter):
    import os
    os.environ["SCRAPERAPI_KEY"] = "fe3f5ba5f4aa0db8f2175ab3afd80341"
    %run scrape_carrefour_full.py

How product data is found:
    Carrefour's site is a Next.js app using React Server Components. Product
    data isn't in a clean <script id="__NEXT_DATA__"> tag -- it's streamed in
    via self.__next_f.push([...]) chunks. One of these chunks (size varies,
    don't assume a fixed index) contains a JSON-like blob with all products
    for that category page, with fields like productId, productName,
    sellingPrice, markedPrice, currency, productType, productUrl.
    There is NO dedicated size/weight/unit field -- pack size (e.g. "14g x 10
    Pieces") is only present inside productName, so it's extracted via regex.
"""


API_KEY = os.environ.get("SCRAPERAPI_KEY")
if not API_KEY:
    raise SystemExit(
        "Missing SCRAPERAPI_KEY. Set it first:\n"
        '  os.environ["SCRAPERAPI_KEY"] = "fe3f5ba5f4aa0db8f2175ab3afd80341"'
    )

BASE_URL = "http://api.scraperapi.com"
OUTPUT_JSON = "carrefour_products.json"
OUTPUT_CSV = "carrefour_products.csv"

# Known category ID -> name mapping (product JSON only has the ID, not a
# readable name, so this has to be supplied ourselves).
CATEGORY_NAMES = {
    "FKEN1700000": "Food Cupboard",
    "FKEN1600000": "Fresh Food",
    "FKEN1660000": "Fruits & Vegetables",
    "FKEN1500000": "Beverages",
    "NFKEN3000000": "Cleaning & Household",
    "FKEN1000000": "Baby Products",
    "FKEN1200000": "Bio & Organic Food",
    "NFKEN2000000": "Beauty & Personal Care",
    "FKEN21000000": "Uncategorized (FKEN21000000)",
    "FKEN6000000": "Uncategorized (FKEN6000000)",
    "FKEN1610000": "Uncategorized (FKEN1610000)",
    "NFKEN8000000": "Uncategorized (NFKEN8000000)",
    "FKEN1100000": "Uncategorized (FKEN1100000)",
}

CATEGORY_URLS = [f"https://www.carrefour.ke/mafken/en/c/{cid}" for cid in CATEGORY_NAMES]

SECONDS_BETWEEN_REQUESTS = 5

# --- Regex patterns -----------------------------------------------------

NEXT_F_CHUNK_PATTERN = re.compile(
    r'self\.__next_f\.push\(\[\d+,(.*?)\]\)\s*</script>', re.DOTALL
)

PRODUCT_PATTERN = re.compile(
    r'\\"productId\\":\\"(?P<productId>[^"\\]+)\\",\\"productName\\":\\"(?P<productName>[^"\\]+)\\".*?'
    r'\\"sellingPrice\\":(?P<sellingPrice>[\d.]+).*?'
    r'\\"markedPrice\\":(?P<markedPrice>[\d.]+).*?'
    r'\\"currency\\":\\"(?P<currency>[^"\\]+)\\".*?'
    r'\\"productType\\":\\"(?P<productType>[^"\\]+)\\".*?'
    r'\\"productUrl\\":\\"(?P<productUrl>[^"\\]+)\\"',
    re.DOTALL,
)

UNIT_PATTERN = re.compile(
    r'(\d+(?:\.\d+)?\s?(?:g|kg|ml|l|Ltrs?|litres?)\s?x?\s?\d*\s?(?:pieces?|pcs?|pack)?)',
    re.IGNORECASE,
)


# --- Core functions -----------------------------------------------------

def fetch_rendered_html(url: str) -> str:
    params = {"api_key": API_KEY, "url": url, "render": "true"}
    resp = requests.get(BASE_URL, params=params, timeout=90)
    resp.raise_for_status()
    return resp.text


def extract_products_from_html(html: str, category_id: str, category_name: str) -> list:
    """
    Search ALL __next_f chunks (not just a fixed index) for product data,
    since the chunk index containing products can vary between pages.
    """
    chunks = NEXT_F_CHUNK_PATTERN.findall(html)

    all_matches = []
    for chunk in chunks:
        matches = list(PRODUCT_PATTERN.finditer(chunk))
        if matches:
            all_matches.extend(matches)

    products = []
    seen_ids = set()
    for m in all_matches:
        d = m.groupdict()
        if d["productId"] in seen_ids:
            continue  # a product could theoretically appear in more than one chunk
        seen_ids.add(d["productId"])

        unit_match = UNIT_PATTERN.search(d["productName"])
        products.append({
            "product_id": d["productId"],
            "description": d["productName"],
            "price": float(d["sellingPrice"]),
            "marked_price": float(d["markedPrice"]),
            "currency": d["currency"],
            "product_type": d["productType"],
            "unit": unit_match.group(1).strip() if unit_match else None,
            "product_url": "https://www.carrefour.ke" + d["productUrl"]
                            if d["productUrl"].startswith("/") else d["productUrl"],
            "category_id": category_id,
            "category": category_name,
        })

    return products


def load_existing() -> dict:
    """Load existing products keyed by product_id, for resume support."""
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, "r") as f:
            data = json.load(f)
        return {p["product_id"]: p for p in data}
    return {}


def save_json(products_by_id: dict):
    with open(OUTPUT_JSON, "w") as f:
        json.dump(list(products_by_id.values()), f, indent=2)


def save_csv(products_by_id: dict):
    import csv
    if not products_by_id:
        return
    fieldnames = ["product_id", "description", "price", "marked_price", "currency",
                  "product_type", "unit", "product_url", "category_id", "category"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(products_by_id.values())


def load_scraped_categories() -> set:
    """Track which category URLs have already been fully scraped, separate
    from the product-level dedup, so we don't re-fetch a whole category page
    unnecessarily on resume."""
    path = "scraped_categories.json"
    if os.path.exists(path):
        with open(path) as f:
            return set(json.load(f))
    return set()


def save_scraped_categories(done: set):
    with open("scraped_categories.json", "w") as f:
        json.dump(sorted(done), f)


def main():
    products_by_id = load_existing()
    scraped_categories = load_scraped_categories()

    remaining = [u for u in CATEGORY_URLS if u not in scraped_categories]
    print(f"{len(remaining)}/{len(CATEGORY_URLS)} categories left to scrape.")
    print(f"{len(products_by_id)} products already saved from prior runs.\n")

    for i, url in enumerate(remaining, 1):
        category_id = url.rstrip("/").split("/")[-1]
        category_name = CATEGORY_NAMES.get(category_id, category_id)
        print(f"[{i}/{len(remaining)}] Scraping: {url} ({category_name})")

        try:
            html = fetch_rendered_html(url)
            new_products = extract_products_from_html(html, category_id, category_name)

            added = 0
            for p in new_products:
                if p["product_id"] not in products_by_id:
                    added += 1
                products_by_id[p["product_id"]] = p  # overwrite/update in place

            save_json(products_by_id)
            save_csv(products_by_id)

            scraped_categories.add(url)
            save_scraped_categories(scraped_categories)

            print(f"  Found {len(new_products)} products ({added} new). "
                  f"Total unique products so far: {len(products_by_id)}")

            if len(new_products) == 0:
                print("  WARNING: 0 products extracted -- page structure may "
                      "have changed, or this category is genuinely empty.")

        except requests.HTTPError as e:
            print(f"  HTTP error: {e.response.status_code} - {e.response.text[:300]}")
        except Exception as e:
            print(f"  Error: {e}")

        if i < len(remaining):
            time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"\nDone. {len(products_by_id)} total unique products saved to "
          f"{OUTPUT_JSON} and {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

0/13 categories left to scrape.
794 products already saved from prior runs.


Done. 794 total unique products saved to carrefour_products.json and carrefour_products.csv


In [ ]:
"""
Carrefour full category scraper using ScraperAPI + regex extraction.

Setup:
    pip install requests beautifulsoup4

Usage (Jupyter):
    import os
    os.environ["SCRAPERAPI_KEY"] = "fe3f5ba5f4aa0db8f2175ab3afd80341"
    %run scrape_carrefour_full.py

How product data is found:
    Carrefour's site is a Next.js app using React Server Components. Product
    data isn't in a clean <script id="__NEXT_DATA__"> tag -- it's streamed in
    via self.__next_f.push([...]) chunks. One of these chunks (size varies,
    don't assume a fixed index) contains a JSON-like blob with all products
    for that category page, with fields like productId, productName,
    sellingPrice, markedPrice, currency, productType, productUrl.
    There is NO dedicated size/weight/unit field -- pack size (e.g. "14g x 10
    Pieces") is only present inside productName, so it's extracted via regex.
"""


API_KEY = os.environ.get("SCRAPERAPI_KEY")
if not API_KEY:
    raise SystemExit(
        "Missing SCRAPERAPI_KEY. Set it first:\n"
        '  os.environ["SCRAPERAPI_KEY"] = "fe3f5ba5f4aa0db8f2175ab3afd80341"'
    )

BASE_URL = "http://api.scraperapi.com"
OUTPUT_JSON = "carrefour_products.json"
OUTPUT_CSV = "carrefour_products.csv"

# Known category ID -> name mapping (product JSON only has the ID, not a
# readable name, so this has to be supplied ourselves).
CATEGORY_NAMES = {
    "FKEN1700000": "Food Cupboard",
    "FKEN1600000": "Fresh Food",
    "FKEN1660000": "Fruits & Vegetables",
    "FKEN1500000": "Beverages",
    "NFKEN3000000": "Cleaning & Household",
    "FKEN1000000": "Baby Products",
    "FKEN1200000": "Bio & Organic Food",
    "NFKEN2000000": "Beauty & Personal Care",
    "FKEN21000000": "Uncategorized (FKEN21000000)",
    "FKEN6000000": "Uncategorized (FKEN6000000)",
    "FKEN1610000": "Uncategorized (FKEN1610000)",
    "NFKEN8000000": "Uncategorized (NFKEN8000000)",
    "FKEN1100000": "Uncategorized (FKEN1100000)",
}

CATEGORY_URLS = [f"https://www.carrefour.ke/mafken/en/c/{cid}" for cid in CATEGORY_NAMES]

SECONDS_BETWEEN_REQUESTS = 5

# --- Regex patterns -----------------------------------------------------

NEXT_F_CHUNK_PATTERN = re.compile(
    r'self\.__next_f\.push\(\[\d+,(.*?)\]\)\s*</script>', re.DOTALL
)

PRODUCT_PATTERN = re.compile(
    r'\\"productId\\":\\"(?P<productId>[^"\\]+)\\",\\"productName\\":\\"(?P<productName>[^"\\]+)\\".*?'
    r'\\"sellingPrice\\":(?P<sellingPrice>[\d.]+).*?'
    r'\\"markedPrice\\":(?P<markedPrice>[\d.]+).*?'
    r'\\"currency\\":\\"(?P<currency>[^"\\]+)\\".*?'
    r'\\"productType\\":\\"(?P<productType>[^"\\]+)\\".*?'
    r'\\"productUrl\\":\\"(?P<productUrl>[^"\\]+)\\"',
    re.DOTALL,
)

UNIT_PATTERN = re.compile(
    r'(\d+(?:\.\d+)?\s?(?:g|kg|ml|l|Ltrs?|litres?)\s?x?\s?\d*\s?(?:pieces?|pcs?|pack)?)',
    re.IGNORECASE,
)


# --- Core functions -----------------------------------------------------

def fetch_rendered_html(url: str) -> str:
    params = {"api_key": API_KEY, "url": url, "render": "true"}
    resp = requests.get(BASE_URL, params=params, timeout=90)
    resp.raise_for_status()
    return resp.text


def scrape_category_all_pages(base_url: str, category_id: str, category_name: str,
                               max_pages: int = 30) -> list:
    """
    Paginate through a category, since a single page load only returns the
    first batch of products (~60), not the full category. Stops when a page
    returns zero NEW products (not just zero products, in case of a blip),
    or when max_pages is hit as a safety cap.
    """
    all_products = {}
    consecutive_empty = 0

    for page in range(0, max_pages):
        # Carrefour's own API used currentPage=N (0-indexed) -- try that as
        # a query param on the category page URL.
        paged_url = f"{base_url}?currentPage={page}"
        print(f"    page {page}: {paged_url}")

        try:
            html = fetch_rendered_html(paged_url)
        except requests.HTTPError as e:
            print(f"    HTTP error on page {page}: {e.response.status_code}")
            break
        except Exception as e:
            print(f"    Error on page {page}: {e}")
            break

        page_products = extract_products_from_html(html, category_id, category_name)
        new_on_this_page = [p for p in page_products if p["product_id"] not in all_products]

        print(f"    -> {len(page_products)} products on page, "
              f"{len(new_on_this_page)} new")

        if not new_on_this_page:
            consecutive_empty += 1
            if consecutive_empty >= 2:  # two empty pages in a row = genuinely done
                break
        else:
            consecutive_empty = 0
            for p in new_on_this_page:
                all_products[p["product_id"]] = p

        time.sleep(SECONDS_BETWEEN_REQUESTS)

    return list(all_products.values())


def extract_products_from_html(html: str, category_id: str, category_name: str) -> list:
    """
    Search ALL __next_f chunks (not just a fixed index) for product data,
    since the chunk index containing products can vary between pages.
    """
    chunks = NEXT_F_CHUNK_PATTERN.findall(html)

    all_matches = []
    for chunk in chunks:
        matches = list(PRODUCT_PATTERN.finditer(chunk))
        if matches:
            all_matches.extend(matches)

    products = []
    seen_ids = set()
    for m in all_matches:
        d = m.groupdict()
        if d["productId"] in seen_ids:
            continue  # a product could theoretically appear in more than one chunk
        seen_ids.add(d["productId"])

        unit_match = UNIT_PATTERN.search(d["productName"])
        products.append({
            "product_id": d["productId"],
            "description": d["productName"],
            "price": float(d["sellingPrice"]),
            "marked_price": float(d["markedPrice"]),
            "currency": d["currency"],
            "product_type": d["productType"],
            "unit": unit_match.group(1).strip() if unit_match else None,
            "product_url": "https://www.carrefour.ke" + d["productUrl"]
                            if d["productUrl"].startswith("/") else d["productUrl"],
            "category_id": category_id,
            "category": category_name,
        })

    return products


def load_existing() -> dict:
    """Load existing products keyed by product_id, for resume support."""
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, "r") as f:
            data = json.load(f)
        return {p["product_id"]: p for p in data}
    return {}


def save_json(products_by_id: dict):
    with open(OUTPUT_JSON, "w") as f:
        json.dump(list(products_by_id.values()), f, indent=2)


def save_csv(products_by_id: dict):
    import csv
    if not products_by_id:
        return
    fieldnames = ["product_id", "description", "price", "marked_price", "currency",
                  "product_type", "unit", "product_url", "category_id", "category"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(products_by_id.values())


def load_scraped_categories() -> set:
    """Track which category URLs have already been fully scraped, separate
    from the product-level dedup, so we don't re-fetch a whole category page
    unnecessarily on resume."""
    path = "scraped_categories.json"
    if os.path.exists(path):
        with open(path) as f:
            return set(json.load(f))
    return set()


def save_scraped_categories(done: set):
    with open("scraped_categories.json", "w") as f:
        json.dump(sorted(done), f)


def main():
    products_by_id = load_existing()

    print(f"{len(products_by_id)} products already saved from prior runs.")
    print(f"Re-scraping all {len(CATEGORY_URLS)} categories WITH PAGINATION "
          f"(this replaces the old one-page-only, skip-if-done behavior).\n")

    for i, url in enumerate(CATEGORY_URLS, 1):
        category_id = url.rstrip("/").split("/")[-1]
        category_name = CATEGORY_NAMES.get(category_id, category_id)
        print(f"[{i}/{len(CATEGORY_URLS)}] Scraping: {url} ({category_name})")

        try:
            new_products = scrape_category_all_pages(url, category_id, category_name)

            added = 0
            for p in new_products:
                if p["product_id"] not in products_by_id:
                    added += 1
                products_by_id[p["product_id"]] = p  # overwrite/update in place

            save_json(products_by_id)
            save_csv(products_by_id)

            print(f"  Category total: {len(new_products)} products ({added} new). "
                  f"Grand total unique products so far: {len(products_by_id)}\n")

            if len(new_products) == 0:
                print("  WARNING: 0 products extracted across all pages -- page "
                      "structure may have changed, or this category is genuinely "
                      "empty.\n")

        except Exception as e:
            print(f"  Error on category {category_name}: {e}\n")

    print(f"\nDone. {len(products_by_id)} total unique products saved to "
          f"{OUTPUT_JSON} and {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

4193 products already saved from prior runs.
Re-scraping all 13 categories WITH PAGINATION (this replaces the old one-page-only, skip-if-done behavior).

[1/13] Scraping: https://www.carrefour.ke/mafken/en/c/FKEN1700000 (Food Cupboard)
    page 0: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=0
    -> 64 products on page, 64 new
    page 1: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=1
    -> 65 products on page, 59 new
    page 2: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=2
    -> 65 products on page, 59 new
    page 3: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=3
    -> 66 products on page, 60 new
    page 4: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=4
    -> 64 products on page, 58 new
    page 5: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=5
    -> 66 products on page, 60 new
    page 6: https://www.carrefour.ke/mafken/en/c/FKEN1700000?currentPage=6
    -> 65 products on page, 5